# 1. Objective and Leakage Rules

This notebook creates a small set of interpretable features for predicting `slowdown_in_5min`. It uses only the current and past observations within each machine, run, and segment, and it does not change the existing labels or splits.

The current label definition and validation split still have review points, including the full-run context-switch baseline and the small validation set. This notebook does not modify those previous stages, so resulting model experiments should remain provisional until the team confirms them.

# 2. Imports and Configuration

Only pandas and NumPy are needed. The grouping columns keep temporal calculations inside one continuous segment, while the target and forbidden columns are never used to create features.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

GROUP_COLUMNS = ["machine_id", "run_id", "segment_id"]
TIME_COLUMN = "timestamp"
TARGET_COLUMN = "slowdown_in_5min"
REQUIRED_COLUMNS = ["id", *GROUP_COLUMNS, TIME_COLUMN, TARGET_COLUMN]
MAIN_METRICS = ["cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"]


# 3. Input and Output Paths

The three original split files are read independently. Feature datasets use new filenames, so the source splits are never overwritten.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

INPUT_PATHS = {
    "train": PROJECT_ROOT / "data/processed/train_dataset.csv",
    "validation": PROJECT_ROOT / "data/processed/val_dataset.csv",
    "test": PROJECT_ROOT / "data/processed/test_dataset.csv",
}
OUTPUT_PATHS = {
    "train": PROJECT_ROOT / "data/processed/train_features.csv",
    "validation": PROJECT_ROOT / "data/processed/val_features.csv",
    "test": PROJECT_ROOT / "data/processed/test_features.csv",
}

split_data = {name: pd.read_csv(path) for name, path in INPUT_PATHS.items()}


# 4. Dataset Inspection

This inspection confirms the size, time range, identifiers, missing values, and target balance of each split. Only the information needed before feature creation is displayed.

In [ ]:
for split_name, data in split_data.items():
    parsed_time = pd.to_datetime(data[TIME_COLUMN], format="mixed", errors="coerce", utc=True)
    print(f"\n{split_name.upper()} — {INPUT_PATHS[split_name]}")
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())
    display(pd.DataFrame({"data_type": data.dtypes.astype(str), "missing": data.isna().sum()}))
    display(data[TARGET_COLUMN].value_counts().sort_index().rename("count").to_frame())
    print("Machines:", data["machine_id"].nunique())
    print("Runs:", data["run_id"].nunique())
    print("Segments:", data["segment_id"].nunique())
    print("Time range:", parsed_time.min(), "to", parsed_time.max())


# 5. Required Column Validation

All splits must contain the audit identifiers, timestamp, and target and must have the same schema. Feature metrics are optional sensors: a missing metric is reported and skipped instead of stopping the complete pipeline.

In [ ]:
for split_name, data in split_data.items():
    missing_required = [column for column in REQUIRED_COLUMNS if column not in data.columns]
    if missing_required:
        raise ValueError(f"{split_name} is missing required columns: {missing_required}")

reference_schema = split_data["train"].columns.tolist()
for split_name in ["validation", "test"]:
    if split_data[split_name].columns.tolist() != reference_schema:
        raise ValueError(f"{split_name} does not have the same schema as train")

available_main_metrics = [column for column in MAIN_METRICS if column in reference_schema]
skipped_main_metrics = [column for column in MAIN_METRICS if column not in reference_schema]
if skipped_main_metrics:
    print("Warning — skipped optional metrics:", skipped_main_metrics)
print("Available main metrics:", available_main_metrics)


# 6. Minimal Feature Definitions

The feature set is deliberately small and fixed. It contains six temporal features per main metric, three domain features, and up to four missing-sensor indicators.

### Previous-value feature

The one-step lag represents the immediately previous system state. It uses only the prior row in the same segment and is missing at each segment start.

### First difference

The first difference measures the change from the previous value to the current value. It uses the same past-only group boundary and is also missing at each segment start.

### Trailing rolling mean

The 30- and 60-second means represent short and persistent pressure. They use actual timestamps, current and past rows only, and require at least three observations.

### Trailing rolling standard deviation

The 60-second standard deviation measures recent instability. It uses the same trailing segment-only history and is missing until three observations are available.

### Trailing rolling maximum

The 60-second maximum captures recent spikes that a mean may hide. It uses only current and earlier rows in the same segment and requires at least three observations.

### Domain features

CPU/RAM interaction represents simultaneous pressure, threads per process represents workload intensity, and time since segment start represents monitoring duration. They use only current values or the first timestamp in the same segment and may remain missing when a source value or denominator is unavailable.

### Missing-value indicators

Four indicators record whether selected sensors are unavailable at the current row. The original values remain unchanged and no imputation is performed.

In [ ]:
AUDIT_COLUMNS = ["id", *GROUP_COLUMNS, TIME_COLUMN, TARGET_COLUMN]
SAFE_RAW_COLUMNS = [
    "elapsed_seconds", "cpu_pct", "cpu_frequency_mhz",
    "ram_pct", "ram_used_mb", "ram_available_mb",
    "swap_pct", "swap_used_mb", "disk_usage_pct", "disk_free_gb",
    "disk_read_mb_s", "disk_write_mb_s", "disk_latency_ms",
    "net_sent_mb_s", "net_recv_mb_s", "network_latency_ms",
    "process_count", "thread_count", "context_switches_per_s",
    "temperature_c", "gpu_usage_pct", "battery_pct",
    "battery_plugged", "missed_deadline", "sample_reliable",
]
MISSING_INDICATOR_SOURCES = [
    "network_latency_ms", "context_switches_per_s",
    "temperature_c", "gpu_usage_pct",
]
FORBIDDEN_OUTPUT_COLUMNS = [
    "slowdown_now", "ended_at_utc", "run_complete", "status", "phase",
    "legacy_id", "stress_cpu_target_pct", "stress_memory_target_mb",
    "cpu_per_core_json", "gpu_per_device_json", "sensor_errors_json",
]


# 7. Reusable Feature Engineering Function

This single function is used unchanged for every split. It sorts rows, creates all temporal features inside each segment, selects the safe output columns, and verifies row, target, identity, and timestamp preservation before returning data.

In [ ]:
def create_features(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    missing_required = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_required:
        raise ValueError(f"{split_name} is missing required columns: {missing_required}")
    if df["id"].duplicated().any():
        raise ValueError(f"{split_name} contains duplicated IDs")

    original = df.copy()
    data = df.copy()
    data[TIME_COLUMN] = pd.to_datetime(data[TIME_COLUMN], format="mixed", errors="coerce", utc=True)
    if data[TIME_COLUMN].isna().any():
        raise ValueError(f"{split_name} contains invalid timestamps")
    data = data.sort_values([*GROUP_COLUMNS, TIME_COLUMN, "id"]).reset_index(drop=True)

    metrics = [column for column in MAIN_METRICS if column in data.columns]
    new_features = []
    grouped = data.groupby(GROUP_COLUMNS, sort=False)

    for metric in metrics:
        lag_name = f"{metric}_lag_1"
        diff_name = f"{metric}_diff_1"
        data[lag_name] = grouped[metric].shift(1)
        data[diff_name] = grouped[metric].diff(1)
        new_features.extend([lag_name, diff_name])

        for feature_name in [f"{metric}_mean_30s", f"{metric}_mean_60s", f"{metric}_std_60s", f"{metric}_max_60s"]:
            data[feature_name] = np.nan
            new_features.append(feature_name)

    for _, segment in data.groupby(GROUP_COLUMNS, sort=False):
        time_indexed = segment.set_index(TIME_COLUMN)
        for metric in metrics:
            data.loc[segment.index, f"{metric}_mean_30s"] = time_indexed[metric].rolling("30s", min_periods=3).mean().to_numpy()
            rolling_60s = time_indexed[metric].rolling("60s", min_periods=3)
            data.loc[segment.index, f"{metric}_mean_60s"] = rolling_60s.mean().to_numpy()
            data.loc[segment.index, f"{metric}_std_60s"] = rolling_60s.std().to_numpy()
            data.loc[segment.index, f"{metric}_max_60s"] = rolling_60s.max().to_numpy()

    if {"cpu_pct", "ram_pct"}.issubset(data.columns):
        data["cpu_ram_interaction"] = (data["cpu_pct"] / 100.0) * (data["ram_pct"] / 100.0)
        new_features.append("cpu_ram_interaction")
    if {"thread_count", "process_count"}.issubset(data.columns):
        data["threads_per_process"] = data["thread_count"] / data["process_count"].replace(0, np.nan)
        new_features.append("threads_per_process")

    segment_start = data.groupby(GROUP_COLUMNS)[TIME_COLUMN].transform("min")
    data["time_since_segment_start_seconds"] = (data[TIME_COLUMN] - segment_start).dt.total_seconds()
    new_features.append("time_since_segment_start_seconds")

    for column in MISSING_INDICATOR_SOURCES:
        if column in data.columns:
            indicator_name = f"{column}_missing"
            data[indicator_name] = data[column].isna().astype("int8")
            new_features.append(indicator_name)

    original_by_id = original.set_index("id").sort_index()
    data_by_id = data.set_index("id").sort_index()
    if len(data) != len(original) or set(data["id"]) != set(original["id"]):
        raise AssertionError(f"{split_name}: rows or IDs changed")
    for column in [TARGET_COLUMN, *GROUP_COLUMNS]:
        if not original_by_id[column].equals(data_by_id[column]):
            raise AssertionError(f"{split_name}: {column} changed")
    original_time = pd.to_datetime(original_by_id[TIME_COLUMN], format="mixed", utc=True)
    if not original_time.equals(data_by_id[TIME_COLUMN]):
        raise AssertionError(f"{split_name}: timestamps changed")

    first_rows = data.groupby(GROUP_COLUMNS, sort=False).head(1)
    for metric in metrics:
        if not first_rows[f"{metric}_lag_1"].isna().all():
            raise AssertionError(f"{split_name}: a lag crossed a group boundary")
        if not first_rows[f"{metric}_diff_1"].isna().all():
            raise AssertionError(f"{split_name}: a difference crossed a group boundary")
    if not first_rows["time_since_segment_start_seconds"].eq(0).all():
        raise AssertionError(f"{split_name}: segment start time check failed")

    output_columns = AUDIT_COLUMNS + [column for column in SAFE_RAW_COLUMNS if column in data.columns] + new_features
    output = data[output_columns].copy()
    if set(FORBIDDEN_OUTPUT_COLUMNS) & set(output.columns):
        raise AssertionError(f"{split_name}: a forbidden column reached the output")
    if np.isinf(output.select_dtypes(include=np.number).to_numpy()).any():
        raise AssertionError(f"{split_name}: infinite numeric values found")
    return output


# 8. Train Feature Engineering

The reusable function is applied to train by itself. No validation or test rows are available to its group calculations.

In [ ]:
train_features = create_features(split_data["train"], "train")
print("Train shape:", train_features.shape)


# 9. Validation Feature Engineering

Validation is processed independently with the same deterministic function. Its observations cannot provide history to train or test.

In [ ]:
validation_features = create_features(split_data["validation"], "validation")
print("Validation shape:", validation_features.shape)


# 10. Test Feature Engineering

Test is processed last and independently. It is not used to choose features, windows, or missing-value treatments.

In [ ]:
test_features = create_features(split_data["test"], "test")
print("Test shape:", test_features.shape)


# 11. Leakage and Integrity Checks

These checks confirm identical output schemas, unchanged targets and identifiers, missing lag/difference values at group starts, zero elapsed time at segment starts, and no infinities. The implementation uses only `shift(1)` and trailing group-specific windows; it contains no future shift, centered window, full-run statistic, or split concatenation.

In [ ]:
feature_splits = {
    "train": train_features,
    "validation": validation_features,
    "test": test_features,
}

feature_schema = train_features.columns.tolist()
if any(data.columns.tolist() != feature_schema for data in [validation_features, test_features]):
    raise AssertionError("Feature schemas are not identical")

new_feature_names = [
    column for column in feature_schema
    if column not in AUDIT_COLUMNS and column not in SAFE_RAW_COLUMNS
]
if len(new_feature_names) > 37:
    raise AssertionError("The requested maximum feature set was exceeded")

print("Created features:", new_feature_names)
print("Checks passed: past-only grouped calculations, preserved rows/targets/IDs/timestamps, no forbidden output columns, and no infinities.")

missing_summary = pd.concat(
    {name: data.isna().sum() for name, data in feature_splits.items()}, axis=1
)
display(missing_summary[missing_summary.gt(0).any(axis=1)])

print("Target distributions after feature engineering:")
display(pd.concat({name: data[TARGET_COLUMN].value_counts().sort_index() for name, data in feature_splits.items()}, axis=1))


# 12. Feature Catalogue

The catalogue documents every new feature, its source, and its time direction. It stays in memory and is not saved as another project artifact.

In [ ]:
catalogue_rows = []
for metric in available_main_metrics:
    definitions = [
        (f"{metric}_lag_1", "lag", "1 previous observation", True, "Represents the previous system state"),
        (f"{metric}_diff_1", "difference", "1 previous observation", True, "Measures the latest change"),
        (f"{metric}_mean_30s", "rolling mean", "30 seconds", True, "Measures short-term sustained pressure"),
        (f"{metric}_mean_60s", "rolling mean", "60 seconds", True, "Measures persistent pressure"),
        (f"{metric}_std_60s", "rolling standard deviation", "60 seconds", True, "Measures recent instability"),
        (f"{metric}_max_60s", "rolling maximum", "60 seconds", True, "Captures recent spikes"),
    ]
    for feature_name, family, window, may_be_missing, reason in definitions:
        catalogue_rows.append([feature_name, family, metric, window, True, may_be_missing, reason])

catalogue_rows.extend([
    ["cpu_ram_interaction", "domain", "cpu_pct, ram_pct", "current row", True, True, "Represents simultaneous CPU and RAM pressure"],
    ["threads_per_process", "domain", "thread_count, process_count", "current row", True, True, "Represents thread intensity"],
    ["time_since_segment_start_seconds", "domain", "timestamp", "segment history", True, False, "Represents monitoring duration"],
])
for column in MISSING_INDICATOR_SOURCES:
    if f"{column}_missing" in new_feature_names:
        catalogue_rows.append([f"{column}_missing", "missing indicator", column, "current row", True, False, "Records sensor availability"])

feature_catalogue = pd.DataFrame(catalogue_rows, columns=[
    "feature_name", "feature_family", "source_columns", "window_or_lag",
    "uses_only_current_and_past", "may_create_missing_values", "reason",
])
display(feature_catalogue)


# 13. Final Summary

The summary compares input and output sizes, target balance, missingness, infinities, and duplicate IDs. It also reports the exact number of feature families created.

In [ ]:
summary_rows = []
for split_name, output in feature_splits.items():
    target_counts = output[TARGET_COLUMN].value_counts()
    summary_rows.append({
        "split": split_name,
        "input_rows": len(split_data[split_name]),
        "output_rows": len(output),
        "input_columns": split_data[split_name].shape[1],
        "output_columns": output.shape[1],
        "new_feature_count": len(new_feature_names),
        "machines": output["machine_id"].nunique(),
        "runs": output["run_id"].nunique(),
        "segments": output["segment_id"].nunique(),
        "target_0_count": int(target_counts.get(0, 0)),
        "target_1_count": int(target_counts.get(1, 0)),
        "target_1_percentage": round(output[TARGET_COLUMN].mean() * 100, 4),
        "total_missing_values": int(output.isna().sum().sum()),
        "infinite_values": int(np.isinf(output.select_dtypes(include=np.number).to_numpy()).sum()),
        "duplicate_ids": int(output["id"].duplicated().sum()),
    })

final_summary = pd.DataFrame(summary_rows)
display(final_summary)
print("Temporal features:", sum(feature_catalogue["feature_family"].isin(["lag", "difference", "rolling mean", "rolling standard deviation", "rolling maximum"])))
print("Domain features:", sum(feature_catalogue["feature_family"].eq("domain")))
print("Missing indicators:", sum(feature_catalogue["feature_family"].eq("missing indicator")))
print("Total new features:", len(feature_catalogue))


# 14. Save Feature Datasets

Files are saved only after every earlier validation and integrity check passes. Each output is written without an export index, and none of the original split files is overwritten.

In [ ]:
for split_name, output_path in OUTPUT_PATHS.items():
    feature_splits[split_name].to_csv(output_path, index=False)
    print(f"Saved {split_name}: {output_path}")


# 15. Conclusion and Next Step

The three splits now have the same minimal, past-only feature definitions while remaining separate. `sample_reliable` and `missed_deadline` are preserved for later review; their final model inclusion depends on real prediction-time availability and whether either behaves like a target proxy.

The next stage may design preprocessing using train only, but this notebook intentionally performs no imputation, scaling, encoding, feature selection, balancing, or model training.